In [9]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PyMuPDFReader
import os

# Create required directories
os.makedirs("static", exist_ok=True)
os.makedirs("test_pdfs", exist_ok=True)

print("✅ Directories created/verified")
print(f"Current directory: {os.getcwd()}")

✅ Directories created/verified
Current directory: /Users/akramelmamoun/Desktop/FAST-APP/notebooks


In [10]:
# Check if PDF file exists
pdf_file = "test_pdfs/Rapport Gestion 2023.pdf"
if os.path.exists(pdf_file):
    print(f"✅ Found PDF: {pdf_file}")
else:
    print(f"❌ PDF not found: {pdf_file}")
    print("Available files in test_pdfs:")
    if os.path.exists("test_pdfs"):
        for f in os.listdir("test_pdfs"):
            print(f"  - {f}")

# Setup reader with error handling
try:
    pdf_reader = PyMuPDFReader()
    reader = SimpleDirectoryReader(
        input_dir="test_pdfs",
        required_exts=[".pdf"],
        recursive=False,
        file_extractor={".pdf": pdf_reader}
    )
    print("✅ Reader setup successful")
except Exception as e:
    print(f"❌ Reader setup failed: {e}")

✅ Found PDF: test_pdfs/Rapport Gestion 2023.pdf
✅ Reader setup successful


In [13]:
try:
    documents = reader.load_data()
    print(f"✅ Successfully loaded {len(documents)} documents")
    
    if documents:
        for i, doc in enumerate(documents):
            filename = doc.metadata.get('filename', 'Unknown')
            char_count = len(doc.text)
            print(f"Document {i+1}: {filename} ({char_count:,} characters)")
            
except Exception as e:
    print(f"❌ Error loading documents: {e}")
    documents = []

✅ Successfully loaded 44 documents
Document 1: Unknown (0 characters)
Document 2: Unknown (2 characters)
Document 3: Unknown (463 characters)
Document 4: Unknown (2 characters)
Document 5: Unknown (27 characters)
Document 6: Unknown (2,102 characters)
Document 7: Unknown (2,600 characters)
Document 8: Unknown (2,443 characters)
Document 9: Unknown (2,186 characters)
Document 10: Unknown (1,826 characters)
Document 11: Unknown (929 characters)
Document 12: Unknown (0 characters)
Document 13: Unknown (55 characters)
Document 14: Unknown (1,878 characters)
Document 15: Unknown (1,832 characters)
Document 16: Unknown (2,206 characters)
Document 17: Unknown (1,380 characters)
Document 18: Unknown (2,336 characters)
Document 19: Unknown (2,929 characters)
Document 20: Unknown (1,755 characters)
Document 21: Unknown (1,092 characters)
Document 22: Unknown (2,243 characters)
Document 23: Unknown (3,090 characters)
Document 24: Unknown (3 characters)
Document 25: Unknown (16 characters)
Documen

[Document(id_='5dcb3621-46be-4522-94a0-d8f686f4ce06', embedding=None, metadata={'file_path': '/Users/akramelmamoun/Desktop/FAST-APP/notebooks/test_pdfs/Rapport Gestion 2023.pdf', 'file_name': 'Rapport Gestion 2023.pdf', 'file_type': 'application/pdf', 'file_size': 20436609, 'creation_date': '2025-07-10', 'last_modified_date': '2025-07-10', 'total_pages': 44, 'source': '1'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=None, image_resource=None, audio_resource=None, video_resource=None, text_template='{metadata_str}\n\n{content}'), Document(id_='ba55e39d-befa-4da7-945f-98e77fb1a869', embedding=None, metadata={'file_path': '/Users/akramelmamoun/Desktop/FAST-APP/notebooks/test_pdfs/Rapp

In [14]:
if documents:
    with open("extracted_text.txt", "w", encoding="utf-8") as f:
        f.write(documents[0].text)
    print("✅ Saved extracted text to `extracted_text.txt`")
    
    # Show preview
    preview = documents[0].text[:500]
    print(f"\nPreview of extracted text:\n{preview}...")
else:
    print("❌ No documents to save")

✅ Saved extracted text to `extracted_text.txt`

Preview of extracted text:
...


In [16]:
import fitz  # PyMuPDF
doc = fitz.open("test_pdfs/Rapport Gestion 2023.pdf")
text = "".join(page.get_text() for page in doc)

print(f"✅ Extracted text from PDF: {len(text)} characters")
with open("extracted_text_mupdf.txt", "w", encoding="utf-8") as f:
    f.write(text)
print("✅ Saved MuPDF extracted text to `extracted_text_mupdf.txt`")    

✅ Extracted text from PDF: 72553 characters
✅ Saved MuPDF extracted text to `extracted_text_mupdf.txt`


In [22]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="test_pdfs/Rapport Gestion 2023.pdf", 
    strategy="hi_res",  # Good quality but more predictable
    infer_table_structure=True,  # Skip table analysis for speed
    extract_images=False,  # Skip image extraction
)
text = "\n".join([str(el) for el in elements])

print(f"✅ Extracted text using Unstructured: {len(text)} characters")
with open("extracted_text_unstructured.txt", "w", encoding="utf-8") as f:
    f.write(text)
print("✅ Saved Unstructured extracted text to `extracted_text_unstructured.txt`")  

KeyboardInterrupt: 

In [26]:
def extract_with_pdfplumber(pdf_path):
    """Extract text using pdfplumber with robust error handling and better text extraction"""
    try:
        import pdfplumber
        
        text_parts = []
        with pdfplumber.open(pdf_path) as pdf:
            print(f"📄 PDF has {len(pdf.pages)} pages")
            
            for page_num, page in enumerate(pdf.pages, 1):
                try:
                    # Try multiple extraction methods
                    page_text = None
                    
                    # Method 1: Standard extraction
                    page_text = page.extract_text()
                    
                    # Method 2: If standard fails, try with layout settings
                    if not page_text or len(page_text.strip()) < 50:
                        page_text = page.extract_text(
                            x_tolerance=2,
                            y_tolerance=2,
                            layout=True,
                            x_density=7.25,
                            y_density=13
                        )
                    
                    # Method 3: Extract text from tables if present (FIXED)
                    if not page_text or len(page_text.strip()) < 50:
                        tables = page.extract_tables()
                        if tables:
                            table_text = ""
                            for table in tables:
                                for row in table:
                                    if row:
                                        # Fix: Properly handle cell values that might be lists or None
                                        clean_cells = []
                                        for cell in row:
                                            if cell is None:
                                                clean_cells.append("")
                                            elif isinstance(cell, list):
                                                clean_cells.append(" ".join(str(item) for item in cell if item))
                                            else:
                                                clean_cells.append(str(cell))
                                        table_text += " | ".join(clean_cells) + "\n"
                            page_text = table_text
                    
                    # Method 4: Character-level extraction as last resort
                    if not page_text or len(page_text.strip()) < 50:
                        chars = page.chars
                        if chars:
                            page_text = "".join([char['text'] for char in chars])
                    
                    if page_text and page_text.strip():
                        text_parts.append(f"\n--- PAGE {page_num} ---\n")
                        text_parts.append(page_text.strip())
                        print(f"Page {page_num}: {len(page_text.strip())} characters extracted")
                    else:
                        print(f"⚠️ Page {page_num}: No text extracted")
                    
                    # Log progress for large PDFs
                    if page_num % 10 == 0:
                        print(f"Processed {page_num}/{len(pdf.pages)} pages")
                        
                except Exception as e:
                    print(f"❌ Error extracting page {page_num}: {e}")
                    continue
        
        text = "\n".join(text_parts)
        extracted_pages = len([part for part in text_parts if not part.startswith("\n--- PAGE")])
        print(f"✅ pdfplumber extracted {len(text):,} characters from {extracted_pages} pages")
        return text
        
    except ImportError:
        print("❌ pdfplumber not installed: pip install pdfplumber")
        return ""
    except Exception as e:
        print(f"❌ pdfplumber extraction failed: {e}")
        return ""

# Test the fixed extraction
text = extract_with_pdfplumber("test_pdfs/Rapport Gestion 2023.pdf")
if text:
    with open("extracted_text_pdfplumber_fixed.txt", "w", encoding="utf-8") as f:
        f.write(text)
    print("✅ Saved to extracted_text_pdfplumber_fixed.txt")
    
    # Show a longer preview to see if it's working
    preview = text[:1000]
    print(f"\nPreview of extracted text:\n{preview}...")
else:
    print("❌ Still no text extracted")

📄 PDF has 44 pages
⚠️ Page 1: No text extracted
Page 2: 1 characters extracted
Page 3: 450 characters extracted
Page 4: 1 characters extracted
Page 5: 2618 characters extracted
❌ Error extracting page 6: unsupported format string passed to list.__format__
❌ Error extracting page 7: unsupported format string passed to list.__format__
❌ Error extracting page 8: unsupported format string passed to list.__format__
❌ Error extracting page 9: unsupported format string passed to list.__format__
❌ Error extracting page 10: unsupported format string passed to list.__format__
❌ Error extracting page 11: unsupported format string passed to list.__format__
⚠️ Page 12: No text extracted
Page 13: 50 characters extracted
❌ Error extracting page 14: unsupported format string passed to list.__format__
❌ Error extracting page 15: unsupported format string passed to list.__format__
❌ Error extracting page 16: unsupported format string passed to list.__format__
❌ Error extracting page 17: unsupported form